## Market Regimes

In [1]:
# import libraries
import json
import numpy as np
import pandas as pd
import joblib

In [2]:
# define csv file
rth_df = pd.read_csv("multiasset_5m_rth_features.csv")

print("Shape:", rth_df.shape)
print("Columns:")
print(rth_df.columns.tolist())

Shape: (171692, 29)
Columns:
['symbol', 'timestamp', 'ts_ny', 'date_ny', 'open', 'high', 'low', 'close', 'volume', 'trade_count', 'ret_1', 'ret_3', 'ret_6', 'ret_12', 'ret_18', 'ret_24', 'atr_12', 'atr_pct_12', 'vwap_session', 'vwap_dist', 'vwap_dist_atr', 'vol_ratio_18', 'range_pct', 'body_pct', 'upper_wick_pct', 'lower_wick_pct', 'rv_12', 'tod_sin', 'tod_cos']


In [3]:
# ensure timestamps account for DST
rth_df["ts_ny"] = pd.to_datetime(rth_df["ts_ny"], utc=True)
rth_df.head()

,symbol,timestamp,ts_ny,date_ny,open,high,low,close,volume,trade_count,...,vwap_dist,vwap_dist_atr,vol_ratio_18,range_pct,body_pct,upper_wick_pct,lower_wick_pct,rv_12,tod_sin,tod_cos
0,COIN,2024-02-28 19:00:00+00:00,2024-02-28 19:00:00+00:00,2024-02-28,205.560,206.0450,204.94,205.4800,174393.0,2678.0,...,0.005424,0.492098,0.505886,0.005378,0.000389,0.002360,0.002628,0.007196,-0.935016,-0.354605
1,COIN,2024-02-28 19:05:00+00:00,2024-02-28 19:05:00+00:00,2024-02-28,205.500,205.6337,204.66,204.8300,111220.0,1865.0,...,0.002227,0.210075,0.364770,0.004754,0.003271,0.000653,0.000830,0.006995,-0.960518,-0.278217
2,COIN,2024-02-28 19:10:00+00:00,2024-02-28 19:10:00+00:00,2024-02-28,204.929,205.0621,203.52,203.7050,169909.0,2386.0,...,-0.003258,-0.307072,0.601525,0.007570,0.006009,0.000653,0.000908,0.007090,-0.979791,-0.200026
3,COIN,2024-02-28 19:15:00+00:00,2024-02-28 19:15:00+00:00,2024-02-28,203.750,204.9900,203.52,204.4947,104130.0,1491.0,...,0.000618,0.061944,0.395331,0.007188,0.003642,0.002422,0.001125,0.005988,-0.992709,-0.120537
4,COIN,2024-02-28 19:20:00+00:00,2024-02-28 19:20:00+00:00,2024-02-28,204.550,204.7199,203.45,203.6800,95168.0,1327.0,...,-0.003358,-0.374749,0.387224,0.006235,0.004271,0.000834,0.001129,0.005532,-0.999189,-0.040266


In [4]:
# sort by stock and time stamp
rth_df = rth_df.sort_values(["ts_ny", "symbol"]).reset_index(drop=True)

print("Columns count:", len(rth_df.columns))
print("Symbols:", rth_df["symbol"].unique())
print("Date range:", rth_df["ts_ny"].min(), "to", rth_df["ts_ny"].max())

Columns count: 29
Symbols: <StringArray>
['TSLA', 'NVDA', 'PLTR', 'COIN', 'OKLO']
Length: 5, dtype: str
Date range: 2024-02-28 18:50:00+00:00 to 2026-02-27 16:55:00+00:00


In [6]:
# check for missing features
feature_cols = [
    "ret_1","ret_3","ret_6","ret_12","ret_18","ret_24",
    "atr_12","atr_pct_12",
    "vwap_session","vwap_dist","vwap_dist_atr",
    "vol_ratio_18",
    "range_pct","body_pct","upper_wick_pct","lower_wick_pct",
    "rv_12",
    "tod_sin","tod_cos"
]

missing_cols = [c for c in feature_cols if c not in rth_df.columns]
print("Missing feature columns:", missing_cols)

Missing feature columns: []


In [7]:
# validate price data is in chronological order
per_symbol_sorted = rth_df.groupby("symbol")["ts_ny"].apply(lambda s: s.is_monotonic_increasing)
print("\nEach Stock is in Chrono Order?")
print(per_symbol_sorted)


Each Stock is in Chrono Order?
symbol
COIN    True
NVDA    True
OKLO    True
PLTR    True
TSLA    True
Name: ts_ny, dtype: bool


In [8]:
# drop rows with NaNs features
mask_valid = rth_df[feature_cols].notna().all(axis=1)
print("Valid rows for clustering:", mask_valid.sum(), "/", len(rth_df))
print("Dropped rows due to NaNs:", (~mask_valid).sum())

Valid rows for clustering: 171692 / 171692
Dropped rows due to NaNs: 0


In [9]:
# verify price data range and daily trading sessions
rth_df["ts_ny"] = pd.to_datetime(rth_df["ts_ny"], utc=True)

rth_df["ts_local"] = rth_df["ts_ny"].dt.tz_convert("America/New_York")
rth_df["session_date"] = rth_df["ts_local"].dt.date

print("Calendar date range:", rth_df["session_date"].min(), "->", rth_df["session_date"].max())
print("Unique trading sessions:", rth_df["session_date"].nunique())

Calendar date range: 2024-02-28 -> 2026-02-27
Unique trading sessions: 484


In [10]:
# calculate daily summary features

# ensure stocks are sorted by time stamp
rth_df = rth_df.sort_values(["symbol", "ts_ny"]).reset_index(drop=True)

# calculate return from close to close for each stock
rth_df["ret_cc"] = rth_df.groupby("symbol")["close"].pct_change()

# calculate daily price data for each stock
daily = (
    rth_df
    .groupby(["symbol", "session_date"])
    .agg(
        day_open=("open","first"),
        day_close=("close","last"),
        day_high=("high","max"),
        day_low=("low","min"),
        day_volume=("volume","sum"),
        bars_in_day=("close","size"),
        daily_rv=("ret_cc", lambda x: np.nanstd(x, ddof=0)),
    )
    .reset_index()
)

daily["daily_ret"] = daily["day_close"] / daily["day_open"] - 1.0
daily["daily_abs_ret"] = daily["daily_ret"].abs()
daily["daily_range_pct"] = (daily["day_high"] - daily["day_low"]) / daily["day_open"]
daily["daily_trend_strength"] = daily["daily_ret"].abs() / (daily["daily_range_pct"] + 1e-12)

print("Daily table shape:", daily.shape)
print("\nSample rows:")
print(daily.head())

Daily table shape: (2237, 13)

Sample rows:
  symbol session_date  day_open  day_close  day_high   day_low  day_volume  \
0   COIN   2024-02-28    205.56    200.730   206.045  199.2400   3674553.0   
1   COIN   2024-02-29    206.46    203.420   211.310  193.8800  14412653.0   
2   COIN   2024-03-01    202.70    205.830   206.390  196.0101   8514752.0   
3   COIN   2024-03-04    217.40    228.720   236.460  212.2469  20685818.0   
4   COIN   2024-03-05    230.00    216.774   239.980  215.4000  21519305.0   

   bars_in_day  daily_rv  daily_ret  daily_abs_ret  daily_range_pct  \
0           24  0.003538  -0.023497       0.023497         0.033105   
1           78  0.007627  -0.014724       0.014724         0.084423   
2           78  0.003980   0.015442       0.015442         0.051208   
3           78  0.008353   0.052070       0.052070         0.111376   
4           78  0.007343  -0.057504       0.057504         0.106870   

   daily_trend_strength  
0              0.709772  
1       

### Rule-Based Regime Classification

Define four trading regimes using threshold rules on the macro features computed above.

The 4 Regimes are Trending, Choppy, Range Bound, and Flat.
- Trending: Strong directional moves - wider targets, longer holds
    - conditions_3 > 0.60 and trend_feq >= 0.33
- Choppy: Volatile but non-directional - tight targets, quick exits
    - conditions_3 < 0.40 and rv_impluse > 0 
- Range Bound: Contained oscillation - moderate targets, conservative holds
    - conditions_3 < 0.40 and rv_impulse <= 0
- Flat: Minimal movement - smallest targets or skip
    - rv_impulse , -0.30 and range_ratios < -0.20 and rel_vol < -0.10

In [11]:
# Create rolling macro features
daily = daily.sort_values(["symbol","session_date"]).reset_index(drop=True)

# using 3 day rolling period
N = 3
LONG_N = 10
eps = 1e-8

# short term realized volatility
daily["rv_3"] = (
    daily.groupby("symbol")["daily_rv"]
    .transform(lambda s: s.shift(1).rolling(N).mean())
)

# long term realized volatility
daily["rv_long"] = (
    daily.groupby("symbol")["daily_rv"]
    .transform(lambda s: s.shift(1).rolling(LONG_N).mean())
)

# volatility expansion or contraction
daily["rv_impulse"] = np.log((daily["rv_3"] + eps) / (daily["rv_long"] + eps))

# range
daily["range_3"] = (
    daily.groupby("symbol")["daily_range_pct"]
    .transform(lambda s: s.shift(1).rolling(N).mean())
)

# long term range
daily["range_long"] = (
    daily.groupby("symbol")["daily_range_pct"]
    .transform(lambda s: s.shift(1).rolling(LONG_N).mean())
)

# range expansion or compression
daily["range_ratio"] = np.log((daily["range_3"] + eps) / (daily["range_long"] + eps))

# trend frequency
trend_thresh = 0.6
daily["trend_freq"] = (
    daily.groupby("symbol")["daily_trend_strength"]
    .transform(lambda s: (s.shift(1) > trend_thresh).rolling(N).mean())
)

# directional or chop
daily["sum_ret_3"] = (
    daily.groupby("symbol")["daily_ret"]
    .transform(lambda s: s.shift(1).rolling(N).sum())
)

daily["sum_abs_ret_3"] = (
    daily.groupby("symbol")["daily_abs_ret"]
    .transform(lambda s: s.shift(1).rolling(N).sum())
)

daily["conditions_3"] = daily["sum_ret_3"].abs() / (daily["sum_abs_ret_3"] + eps)

# relative volume vs past 10 days
daily["rel_vol"] = (
    daily.groupby("symbol")["day_volume"]
    .transform(lambda s: np.log(s.shift(1) / (s.shift(1).rolling(LONG_N).median() + eps)))
)

macro_feature_cols = [
    "rv_impulse",
    "range_ratio",
    "trend_freq",
    "conditions_3",
    "rel_vol"
]

print(daily[["symbol", "session_date"] + macro_feature_cols].head(15))

   symbol session_date  rv_impulse  range_ratio  trend_freq  conditions_3  \
0    COIN   2024-02-28         NaN          NaN         NaN           NaN   
1    COIN   2024-02-29         NaN          NaN         NaN           NaN   
2    COIN   2024-03-01         NaN          NaN    0.333333           NaN   
3    COIN   2024-03-04         NaN          NaN    0.333333      0.424497   
4    COIN   2024-03-05         NaN          NaN    0.000000      0.641898   
5    COIN   2024-03-06         NaN          NaN    0.000000      0.080047   
6    COIN   2024-03-07         NaN          NaN    0.000000      0.238468   
7    COIN   2024-03-08         NaN          NaN    0.000000      0.049591   
8    COIN   2024-03-11         NaN          NaN    0.000000      1.000000   
9    COIN   2024-03-12         NaN          NaN    0.333333      0.035393   
10   COIN   2024-03-13    0.050792     0.096355    0.333333      0.186174   
11   COIN   2024-03-14   -0.152740    -0.089752    0.333333      1.000000   

In [12]:
# normalize by stock looking back 30 days

norm_cols = macro_feature_cols.copy()
z_cols = []

Z_LOOKBACK = 30
eps = 1e-8

for col in norm_cols:
    zcol = f"{col}_z"
    daily[zcol] = (
        daily.groupby("symbol")[col]
        .transform(lambda s: (s - s.shift(1).rolling(Z_LOOKBACK).mean()) /
                             (s.shift(1).rolling(Z_LOOKBACK).std() + eps))
    )
    z_cols.append(zcol)

daily.tail(20)

,symbol,session_date,day_open,day_close,day_high,day_low,day_volume,bars_in_day,daily_rv,daily_ret,...,trend_freq,sum_ret_3,sum_abs_ret_3,conditions_3,rel_vol,rv_impulse_z,range_ratio_z,trend_freq_z,conditions_3_z,rel_vol_z
2217,TSLA,2026-01-22,435.190,449.3700,449.5000,432.6293,63597499.0,78,0.001825,0.032583,...,0.333333,-0.004924,0.051407,0.095786,0.112616,0.959168,0.929559,-0.059946,-1.670872,0.483165
2218,TSLA,2026-01-23,447.425,449.0700,452.4300,444.0400,51146282.0,78,0.001852,0.003677,...,0.666667,0.032278,0.079372,0.406673,0.176865,0.804153,0.999414,0.839248,-0.691032,0.683038
2219,TSLA,2026-01-26,445.020,435.2000,445.0500,434.2800,43782576.0,78,0.002705,-0.022066,...,0.333333,0.059502,0.059502,1.000000,-0.041020,0.074529,0.604429,-0.120567,0.929516,-0.150879
2220,TSLA,2026-01-27,437.410,430.9100,437.5200,430.6900,33965748.0,78,0.001275,-0.014860,...,0.666667,0.014194,0.058326,0.243348,-0.184723,0.019257,-0.051125,0.767744,-1.332384,-0.679170
2221,TSLA,2026-01-28,431.910,431.4600,438.2600,430.1000,42249391.0,78,0.001916,-0.001042,...,0.666667,-0.033250,0.040603,0.818901,-0.418231,-0.253718,-1.097607,0.714742,0.346854,-1.507316
2222,TSLA,2026-01-29,437.640,416.4400,440.2299,414.6200,70903529.0,78,0.002825,-0.048442,...,0.666667,-0.037969,0.037969,1.000000,-0.199994,-0.259478,-1.042554,0.663278,0.865322,-0.649539
2223,TSLA,2026-01-30,425.770,430.6200,439.8800,422.7000,74741961.0,78,0.003738,0.011391,...,0.666667,-0.064344,0.064344,1.000000,0.306076,-0.197647,0.377955,0.612892,0.835249,1.263686
2224,TSLA,2026-02-02,421.250,421.7850,427.1500,414.5000,51665167.0,78,0.004390,0.001270,...,0.333333,-0.038092,0.060875,0.625751,0.338332,1.020240,1.046097,-0.430620,-0.257366,1.455297
2225,TSLA,2026-02-03,424.270,421.8100,428.5600,413.6901,52032872.0,78,0.002471,-0.005798,...,0.333333,-0.035780,0.061103,0.585578,-0.015418,1.876163,1.416954,-0.430620,-0.339737,0.122000
2226,TSLA,2026-02-04,420.400,405.8300,423.9000,399.1800,67848231.0,78,0.003329,-0.034657,...,0.000000,0.006863,0.018459,0.371788,0.003540,1.812003,0.524653,-1.395197,-1.066138,0.268624


In [13]:
# drop NaNs
macro_z_cols = ["rv_impulse","range_ratio","trend_freq","conditions_3","rel_vol",
                "rv_impulse_z","range_ratio_z","trend_freq_z","conditions_3_z","rel_vol_z"]
macro = daily.dropna(subset=macro_z_cols).copy()

print(macro.head())

   symbol session_date  day_open  day_close  day_high  day_low  day_volume  \
40   COIN   2024-04-25    216.25     223.61    225.94   213.64   4451838.0   
41   COIN   2024-04-26    220.77     236.44    237.02   218.66   5472576.0   
42   COIN   2024-04-29    229.94     218.14    230.32   216.54   8487485.0   
43   COIN   2024-04-30    214.67     204.02    216.57   202.59   7891047.0   
44   COIN   2024-05-01    199.00     209.93    218.52   198.20   8957379.0   

    bars_in_day  daily_rv  daily_ret  ...  trend_freq  sum_ret_3  \
40           78  0.005675   0.034035  ...    1.000000   0.047609   
41           78  0.004153   0.070979  ...    0.666667   0.038390   
42           78  0.005703  -0.051318  ...    0.666667   0.052106   
43           78  0.004589  -0.049611  ...    0.666667   0.053696   
44           78  0.005568   0.054925  ...    1.000000  -0.029950   

    sum_abs_ret_3  conditions_3   rel_vol  rv_impulse_z  range_ratio_z  \
40       0.153423      0.310310 -0.359211     -1

In [14]:
entries = pd.read_csv("multiasset_labeled_entries.csv")
entries["ts_entry"] = pd.to_datetime(entries["ts_entry"], errors="coerce")
entries["ts_entry"] = entries["ts_entry"].dt.tz_localize("America/New_York").dt.tz_convert("UTC")

min_val_ts  = entries.loc[entries["split"]=="val", "ts_entry"].min()
min_test_ts = entries.loc[entries["split"]=="test", "ts_entry"].min()

first_val_day  = min_val_ts.tz_convert("America/New_York").date()
first_test_day = min_test_ts.tz_convert("America/New_York").date()

print("first_val_day :", first_val_day)
print("first_test_day:", first_test_day)

first_val_day : 2025-07-15
first_test_day: 2025-10-28


In [15]:
# split macro data
macro_train = macro[macro["session_date"] < first_val_day].copy()
macro_val   = macro[(macro["session_date"] >= first_val_day) & (macro["session_date"] < first_test_day)].copy()
macro_test  = macro[macro["session_date"] >= first_test_day].copy()

print("Macro train rows:", len(macro_train))
print("Macro val rows  :", len(macro_val))
print("Macro test rows :", len(macro_test))

Macro train rows: 1349
Macro val rows  : 335
Macro test rows : 353


#### Threshold Definitions and Regime Assignment

In [16]:
# regime thresholds
T = {
    'flat_rv'     : -0.30,   # rv_impulse: very compressed short-term vol vs long-run
    'flat_range'  : -0.20,   # range_ratio: very narrow range vs history
    'flat_vol'    : -0.10,   # rel_vol: below-average volume
    'trend_cond3' :  0.60,   # conditions_3: strong directional consistency
    'trend_freq'  :  0.33,   # trend_freq: >= 1 of last 3 days was trending
    'chop_cond3'  :  0.40,   # conditions_3: weak directional consistency
    'chop_rv'     :  0.00,   # rv_impulse: short-term vol expanding vs long-run avg
}

# regime labels
def label_regimes(df, t):
    regime = pd.Series('Range Bound', index=df.index, dtype='object')

    choppy   = (df['conditions_3'] < t['chop_cond3'])  & (df['rv_impulse'] > t['chop_rv'])
    trending = (df['conditions_3'] > t['trend_cond3']) & (df['trend_freq'] >= t['trend_freq'])
    flat     = (df['rv_impulse']   < t['flat_rv'])      & (df['range_ratio'] < t['flat_range']) \
             & (df['rel_vol']      < t['flat_vol'])

    regime[choppy]   = 'Choppy'
    regime[trending] = 'Trending'
    regime[flat]     = 'Flat'

    return regime

# apply regime labels
macro['regime_label'] = label_regimes(macro, T)

# encode regimes
REGIME_INT = {'Flat': 0, 'Trending': 1, 'Choppy': 2, 'Range Bound': 3}
macro['macro_regime_id'] = macro['regime_label'].map(REGIME_INT)

print("Regime distribution:")
dist = macro['regime_label'].value_counts()
frac = macro['regime_label'].value_counts(normalize=True).round(3)
print(pd.DataFrame({'count': dist, 'fraction': frac}))

Regime distribution:
              count  fraction
regime_label                 
Range Bound     873     0.429
Trending        710     0.349
Choppy          332     0.163
Flat            122     0.060


In [17]:
# check regime distribution
macro_train_mask = macro['session_date'] < first_val_day
val_mask  = (macro['session_date'] >= first_val_day) & (macro['session_date'] < first_test_day)
test_mask = macro['session_date'] >= first_test_day

for name, mask in [('Train', macro_train_mask), ('Val', val_mask), ('Test', test_mask)]:
    dist = macro.loc[mask, 'regime_label'].value_counts()
    frac = (dist / dist.sum()).round(3)
    print(f"\n{name} set ({mask.sum()} rows):")
    print(pd.DataFrame({'count': dist, 'fraction': frac}))


Train set (1349 rows):
              count  fraction
regime_label                 
Range Bound     569     0.422
Trending        452     0.335
Choppy          233     0.173
Flat             95     0.070

Val set (335 rows):
              count  fraction
regime_label                 
Range Bound     156     0.466
Trending        119     0.355
Choppy           55     0.164
Flat              5     0.015

Test set (353 rows):
              count  fraction
regime_label                 
Range Bound     148     0.419
Trending        139     0.394
Choppy           44     0.125
Flat             22     0.062


In [18]:
# validate regimes
print("Mean macro features per regime:")
display(
    macro.loc[macro_train_mask]
    .groupby('regime_label')[macro_feature_cols]
    .mean()
    .round(4)
    .sort_values('trend_freq', ascending=False)
)

Mean macro features per regime:


,rv_impulse,range_ratio,trend_freq,conditions_3,rel_vol
regime_label,,,,,
Trending,-0.0119,0.0289,0.4373,0.8875,0.1261
Choppy,0.2032,0.0829,0.3648,0.2021,0.1339
Range Bound,-0.1048,-0.0577,0.2613,0.4818,-0.0075
Flat,-0.4694,-0.4327,0.2105,0.5895,-0.3755


In [19]:
# Regime persistence and transitions
macro_sorted = macro.sort_values(['symbol', 'session_date']).copy()
macro_sorted['prev_regime'] = macro_sorted.groupby('symbol')['regime_label'].shift(1)

valid = macro_sorted.dropna(subset=['prev_regime'])
persistence = (valid['regime_label'] == valid['prev_regime']).mean()
print(f"Day-over-day regime persistence: {persistence:.1%}\n")

print("Transition matrix (row = yesterday > col = today):")
trans = pd.crosstab(
    valid['prev_regime'],
    valid['regime_label'],
    normalize='index'
).round(3)
display(trans)

Day-over-day regime persistence: 51.6%

Transition matrix (row = yesterday > col = today):


regime_label,Choppy,Flat,Range Bound,Trending
prev_regime,,,,
Choppy,0.343,0.006,0.410,0.241
Flat,0.016,0.500,0.287,0.197
Range Bound,0.138,0.047,0.561,0.254
Trending,0.133,0.025,0.298,0.544


In [20]:
# save regime thresholds
regime_meta = {
    "thresholds": T,
    "regime_int_map": REGIME_INT,
}
with open("regime_thresholds.json", "w") as f:
    json.dump(regime_meta, f, indent=2)

print("Saved regime_thresholds.json")
print("\nInteger encoding used:")
for label, code in REGIME_INT.items():
    count = (macro['regime_label'] == label).sum()
    print(f"  {code} = {label:<12}  ({count} total session-days)")

Saved regime_thresholds.json

Integer encoding used:
  0 = Flat          (122 total session-days)
  1 = Trending      (710 total session-days)
  2 = Choppy        (332 total session-days)
  3 = Range Bound   (873 total session-days)


In [21]:
# add regime to 5M price data
rth_df = rth_df.merge(
    macro[["symbol", "session_date", "macro_regime_id", "regime_label"]],
    on=["symbol", "session_date"],
    how="left"
)

print("Missing macro_regime_id on bars:", rth_df["macro_regime_id"].isna().sum())
print("Bars shape:", rth_df.shape)
print(rth_df[["symbol", "ts_ny", "session_date", "macro_regime_id", "regime_label"]].head(10))

Missing macro_regime_id on bars: 14656
Bars shape: (171692, 34)
  symbol                     ts_ny session_date  macro_regime_id regime_label
0   COIN 2024-02-28 19:00:00+00:00   2024-02-28              NaN          NaN
1   COIN 2024-02-28 19:05:00+00:00   2024-02-28              NaN          NaN
2   COIN 2024-02-28 19:10:00+00:00   2024-02-28              NaN          NaN
3   COIN 2024-02-28 19:15:00+00:00   2024-02-28              NaN          NaN
4   COIN 2024-02-28 19:20:00+00:00   2024-02-28              NaN          NaN
5   COIN 2024-02-28 19:25:00+00:00   2024-02-28              NaN          NaN
6   COIN 2024-02-28 19:30:00+00:00   2024-02-28              NaN          NaN
7   COIN 2024-02-28 19:35:00+00:00   2024-02-28              NaN          NaN
8   COIN 2024-02-28 19:40:00+00:00   2024-02-28              NaN          NaN
9   COIN 2024-02-28 19:45:00+00:00   2024-02-28              NaN          NaN


In [22]:
# create regime dataset
rth_regime = rth_df.dropna(subset=["macro_regime_id"]).copy()
rth_regime["macro_regime_id"] = rth_regime["macro_regime_id"].astype(int)

print("Regime bars:", rth_regime.shape)
print("Dropped bars:", len(rth_df) - len(rth_regime))

Regime bars: (157036, 34)
Dropped bars: 14656


In [23]:
# add regimes to VWAP Reclaim Long labeled entries csv
entries_df = pd.read_csv("multiasset_labeled_entries.csv")
entries_df["ts_entry"] = pd.to_datetime(entries_df["ts_entry"], errors="coerce")
entries_df["ts_entry"] = entries_df["ts_entry"].dt.tz_localize("America/New_York").dt.tz_convert("UTC")
entries_df["session_date"] = entries_df["ts_entry"].dt.tz_convert("America/New_York").dt.date

entries_df = entries_df.merge(
    macro[["symbol", "session_date", "macro_regime_id", "regime_label"]],
    on=["symbol", "session_date"],
    how="left"
)

print("Entries:", len(entries_df))
print("Entries missing macro_regime_id:", entries_df["macro_regime_id"].isna().sum())

Entries: 1616
Entries missing macro_regime_id: 152


In [24]:
# create entries with regime dataset
entries_regime = entries_df.dropna(subset=["macro_regime_id"]).copy()
entries_regime["macro_regime_id"] = entries_regime["macro_regime_id"].astype(int)

print("Regime entries:", len(entries_regime))

Regime entries: 1464


In [25]:
# save new datasets
rth_regime.to_csv("multiasset_5m_rth_features_with_regime.csv", index=False)
entries_regime.to_csv("multiasset_labeled_entries_with_regime.csv", index=False)

In [26]:
# save daily macro dataset
macro.to_csv("macro_daily_with_regimes.csv", index=False)
print(macro.shape)

(2037, 31)


Now to test Regimes on VWAP Reclaim Setups

In [27]:
# load vwap reclaim entries with regimes file
opps = pd.read_csv("multiasset_labeled_entries_with_regime.csv")
opps["ts_entry"] = pd.to_datetime(opps["ts_entry"], errors="coerce")
opps["ts_entry"] = opps["ts_entry"].dt.tz_convert("UTC")

opps = opps.dropna(subset=["macro_regime_id"]).copy()
opps["macro_regime_id"] = opps["macro_regime_id"].astype(int)

print("Opps shape:", opps.shape)
print(opps["split"].value_counts())
print("\nRegime distribution in entries:")
print(opps["regime_label"].value_counts())

Opps shape: (1464, 36)
split
train    979
test     243
val      242
Name: count, dtype: int64

Regime distribution in entries:
regime_label
Range Bound    633
Trending       500
Choppy         236
Flat            95
Name: count, dtype: int64


best_R is from the simulated results of the VWAP Reclaim setups.

best_R = maximum simulated R-multiple achievable for that setup.

If best_R > 0, at least one of stop/target/hold combination resulted in a profitable trade.

In [28]:
# Win = best_R > 0
# calculate metrics for each regime
opps["win"] = (opps["best_R"] > 0).astype(int)

def regime_perf(df):
    g = df.groupby("regime_label").agg(
        n=("best_R", "size"),
        win_rate=("win", "mean"),
        avg_R=("best_R", "mean"),
        med_R=("best_R", "median"),
        std_R=("best_R", "std"),
        p25_R=("best_R", lambda x: np.percentile(x, 25)),
        p75_R=("best_R", lambda x: np.percentile(x, 75)),
    )
    g["expectancy"] = g["avg_R"]
    return g.sort_values("avg_R", ascending=False)

train_perf = regime_perf(opps[opps["split"] == "train"])
val_perf   = regime_perf(opps[opps["split"] == "val"])
test_perf  = regime_perf(opps[opps["split"] == "test"])

print("\nTrain:")
print(train_perf)
print("\nValidation:")
print(val_perf)
print("\nTest:")
print(test_perf)


Train:
                n  win_rate     avg_R  med_R     std_R  p25_R     p75_R  \
regime_label                                                              
Range Bound   408  0.575980  1.246135    1.5  2.066559   -1.0  3.000000   
Trending      330  0.587879  1.122416    1.5  1.959257   -1.0  2.666667   
Flat           75  0.546667  1.055572    1.5  1.975907   -1.0  2.666667   
Choppy        166  0.506024  0.962549    1.0  2.025942   -1.0  2.666667   

              expectancy  
regime_label              
Range Bound     1.246135  
Trending        1.122416  
Flat            1.055572  
Choppy          0.962549  

Validation:
                n  win_rate     avg_R     med_R     std_R  p25_R   p75_R  \
regime_label                                                               
Range Bound   124  0.620968  1.317440  1.500000  1.998641   -1.0  3.0000   
Choppy         45  0.555556  1.099183  1.333333  2.090782   -1.0  3.0000   
Trending       69  0.550725  0.943403  0.750000  2.039723   -1

In [29]:
opps.groupby(["regime_label", "best_param_id"])["best_R"].mean().unstack()

best_param_id,S0.5_T0.75_H3,S0.5_T0.75_H6,S0.5_T1.0_H12,S0.5_T1.0_H3,S0.5_T1.0_H6,S0.5_T1.5_H12,S0.5_T1.5_H24,S0.5_T1.5_H3,S0.5_T1.5_H6,S0.5_T2.0_H12,...,S1.0_T1.0_H3,S1.0_T1.0_H6,S1.0_T1.5_H12,S1.0_T1.5_H24,S1.0_T1.5_H3,S1.0_T1.5_H6,S1.0_T2.0_H12,S1.0_T2.0_H24,S1.0_T2.0_H3,S1.0_T2.0_H6
regime_label,,,,,,,,,,,,,,,,,,,,,
Choppy,-0.776119,1.500000,2.0,2.0,2.0,NaN,3.0,3.000000,3.000000,4.0,...,1.0,1.0,NaN,NaN,NaN,1.5,2.0,NaN,2.000000,2.0
Flat,-0.891304,NaN,NaN,2.0,NaN,3.0,NaN,3.000000,NaN,4.0,...,1.0,NaN,NaN,NaN,1.5,NaN,2.0,NaN,2.000000,2.0
Range Bound,-0.724631,0.577692,2.0,2.0,2.0,3.0,NaN,2.994247,2.871389,4.0,...,1.0,1.0,1.5,1.067715,1.5,1.5,2.0,2.000000,2.000000,2.0
Trending,-0.681549,0.387866,NaN,2.0,2.0,3.0,3.0,2.992975,3.000000,4.0,...,1.0,NaN,1.5,NaN,1.5,1.5,2.0,1.908291,1.981308,2.0
